In [1]:
# -*- coding: utf-8 -*-
"""Untitled24.ipynb"""

import pandas as pd, numpy as np, scipy.optimize as opt, scipy.stats as stats, math, os

path="/content/PREÇO MED DIARIO PLD VERTICAL PERIODO 17-04-18 A 03-06-25.xlsx"
df=pd.read_excel(path)
df=df.rename(columns={"DATA SUBMERCADO SUDESTE":"data","PREÇO MEDIO DIÁRIO":"preço"})
df["data"]=pd.to_datetime(df["data"])
df=df.sort_values("data").reset_index(drop=True)

df["retorno_log"]=np.log(df["preço"]/df["preço"].shift(1))
df["volatilidade_realizada"]=df["retorno_log"].abs()

r = (df["retorno_log"].dropna().values)*100.0
T=len(r)

def gjr_t_negloglik(theta_raw, r):

    mu = theta_raw[0]
    w_raw, a_raw, g_raw, b_raw, nu_raw = theta_raw[1:]

    omega = np.exp(w_raw)
    alpha = 1/(1+np.exp(-a_raw))
    gamma = 1/(1+np.exp(-g_raw))

    slack = max(1e-6, 1 - alpha - 0.5*gamma)
    beta = (1/(1+np.exp(-b_raw))) * 0.999 * slack

    nu = 2 + np.exp(nu_raw)

    eps = r - mu

    sigma2 = np.empty_like(eps)

    denom = 1 - alpha - 0.5*gamma - beta

    if denom <= 1e-6:
        v0 = np.var(eps)
    else:
        v0 = omega/denom

    sigma2[0] = max(v0, 1e-8)

    for t in range(1, len(eps)):

        ind = 1.0 if eps[t-1] < 0 else 0.0

        sigma2[t] = omega + (alpha + gamma*ind)*(eps[t-1]**2) + beta*sigma2[t-1]

        if sigma2[t] <= 1e-12:
            sigma2[t] = 1e-12

    sigma = np.sqrt(sigma2)
    z = eps / sigma

    scale = math.sqrt(nu/(nu-2))
    u = z * scale

    ll = stats.t.logpdf(u, df=nu) + math.log(scale) - np.log(sigma)

    return -np.sum(ll)


mu0 = np.mean(r)

theta0 = np.array([
    mu0,
    np.log(np.var(r)*0.05 + 1e-6),
    0.0,
    0.0,
    0.0,
    np.log(10.0)
])

res = opt.minimize(
    gjr_t_negloglik,
    theta0,
    args=(r,),
    method="L-BFGS-B",
    options={"maxiter":2000}
)


def fit_gjr_t(r):

    best=None

    starts=[]

    mu0=np.mean(r)

    starts.append(np.array([mu0,np.log(np.var(r)*0.02+1e-6),-1.0,-1.0,1.0,np.log(8.0)]))
    starts.append(np.array([mu0,np.log(np.var(r)*0.05+1e-6),0.5,-0.5,0.5,np.log(15.0)]))
    starts.append(np.array([mu0,np.log(np.var(r)*0.10+1e-6),-0.5,0.5,-0.5,np.log(6.0)]))

    for x0 in starts:

        r1=opt.minimize(gjr_t_negloglik,x0,args=(r,),method="Powell",
                        options={"maxiter":4000})

        x=r1.x

        r2=opt.minimize(gjr_t_negloglik,x,args=(r,),method="L-BFGS-B",
                        options={"maxiter":4000})

        cand=r2 if r2.fun<=r1.fun else r1

        if best is None or cand.fun < best.fun:
            best=cand

    return best


best=fit_gjr_t(r)

mu = best.x[0]
omega = math.exp(best.x[1])
alpha = 1/(1+math.exp(-best.x[2]))
gamma = 1/(1+math.exp(-best.x[3]))

slack = max(1e-6, 1 - alpha - 0.5*gamma)

beta = (1/(1+math.exp(-best.x[4]))) * 0.999 * slack

nu = 2 + math.exp(best.x[5])

phi = alpha + 0.5*gamma + beta


r_full = df["retorno_log"].values*100

eps = r_full - mu

sigma2=np.empty_like(eps)

sigma2[:] = np.nan

sigma2[0]=np.var(eps[~np.isnan(eps)])

for t in range(1,len(eps)):

    if np.isnan(eps[t-1]):

        sigma2[t]=sigma2[t-1]

        continue

    ind=1.0 if eps[t-1] < 0 else 0.0

    sigma2[t]=omega + (alpha + gamma*ind)*(eps[t-1]**2) + beta*sigma2[t-1]


sigma=np.sqrt(sigma2)

df["vol_GJRt_diaria"] = sigma/100.0


# =============================
# NOVA ESCALA TEMPORAL
# =============================

for n,lab in [(5,"1W"),(21,"1M"),(252,"1Y")]:

    df[f"vol_GJRt_{lab}"] = df["vol_GJRt_diaria"] * math.sqrt(n)

    df[f"vol_GJRt_{lab}_anualizada"] = df["vol_GJRt_diaria"] * math.sqrt(252)


out = df[[
"data",
"preço",
"retorno_log",
"volatilidade_realizada",
"vol_GJRt_diaria",
"vol_GJRt_1W",
"vol_GJRt_1M",
"vol_GJRt_1Y",
"vol_GJRt_1W_anualizada",
"vol_GJRt_1M_anualizada",
"vol_GJRt_1Y_anualizada"
]].copy()


out = out.rename(columns={
"preço":"preço_PLD",
"vol_GJRt_diaria":"volatilidade_GJR-t_diária",
"vol_GJRt_1W":"volatilidade_GJR-t_1W",
"vol_GJRt_1M":"volatilidade_GJR-t_1M",
"vol_GJRt_1Y":"volatilidade_GJR-t_1Y",
"vol_GJRt_1W_anualizada":"volatilidade_GJR-t_1W_anualizada",
"vol_GJRt_1M_anualizada":"volatilidade_GJR-t_1M_anualizada",
"vol_GJRt_1Y_anualizada":"volatilidade_GJR-t_1Y_anualizada",
"retorno_log":"retorno_log",
"volatilidade_realizada":"volatilidade_realizada_diária"
})


out_path="/content/PLD_volatilidades_GJR-t.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

    out.to_excel(writer,index=False,sheet_name="GJR-t")

    params=pd.DataFrame({
        "Parâmetro":["mu","omega","alpha","gamma","beta","nu","phi=alpha+0.5*gamma+beta"],
        "Valor":[mu,omega,alpha,gamma,beta,nu,phi]
    })

    params.to_excel(writer,index=False,sheet_name="Parâmetros")


out_path

'/content/PLD_volatilidades_GJR-t.xlsx'